长期低位股票通常呈现以下均线特征：
- **价格长期运行在 MA60/MA120/MA250 下方**
- **短/中/长期均线高度粘合**（波动率压缩）
- **均线呈空头排列**但开始走平

In [ ]:
# 均线压制判断
below_ma60 = df['close'] < df['close'].rolling(60).mean()
below_ma120 = df['close'] < df['close'].rolling(120).mean()
# 连续60天被MA60压制 = 弱势确认
prolonged_low = below_ma60.rolling(60).sum() >= 50

# ─── 均线高度粘合度 ───
ma20 = df['close'].rolling(20).mean()
ma60 = df['close'].rolling(60).mean()
ma120 = df['close'].rolling(120).mean()
ma250 = df['close'].rolling(250).mean()

# 计算三组均线之间的最大间距（归一化），值越小表示越粘合
ma_spread = pd.concat([
    (ma20 - ma60).abs() / ma60,    # 短-中
    (ma60 - ma120).abs() / ma120,  # 中-长
    (ma20 - ma120).abs() / ma120,  # 短-长
], axis=1).max(axis=1)

# 粘合阈值：最大间距 < 5%（可根据标的调整）
is_ma_converged = ma_spread < 0.05

# ─── 波动率压缩（Bollinger Bandwidth） ───
import numpy as np

def bollinger_bandwidth(series, period=20, std_mult=2):
    """布林带宽度 = (上轨-下轨) / 中轨，衡量波动率"""
    ma = series.rolling(period).mean()
    std = series.rolling(period).std()
    bandwidth = 2 * std_mult * std / ma
    return bandwidth

bbw = bollinger_bandwidth(df['close'], period=20)
# 带宽处于过去120天的20%分位以下 = 波动率极度压缩
bbw_low = bbw < bbw.rolling(120).quantile(0.20)

# ─── 均线走平检测（斜率接近零） ───
def ma_slope(series, period=20, lookback=20):
    """
    计算均线斜率（线性回归），返回归一化斜率值。
    斜率 ≈ 0 表示走平，>0 表示上扬，<0 表示下行。
    """
    ma = series.rolling(period).mean()
    # 用过去 lookback 天的 MA 值做线性回归
    x = np.arange(lookback)
    slopes = ma.rolling(lookback).apply(
        lambda y: np.polyfit(x, y, 1)[0] if len(y) == lookback else np.nan,
        raw=True
    )
    # 归一化：斜率 / 当前价格，得到百分比变化，归一化：高价股和低价股乘以k是不同的
    return slopes / ma

slope_ma60 = ma_slope(df['close'], period=60, lookback=20)
slope_ma120 = ma_slope(df['close'], period=120, lookback=30)

# 斜率绝对值 < 0.001（即每天变动不到0.1%）= 走平
is_flattening_60 = slope_ma60.abs() < 0.001
is_flattening_120 = slope_ma120.abs() < 0.001
# 上一期斜率向下（空头排列），本期走平 = 拐点信号
was_bearish_60 = slope_ma60.shift(20) < -0.002
turning_point_60 = was_bearish_60 & is_flattening_60

# 方法1：过去 N 天有足够多 turning_point 就确认
tp_window = df['turning_point_60'].rolling(10).sum()
tp_zone = tp_window >= 3  # 10 天内有 3 天触发就认为是走平区域

# 方法2：连续 N 天中首次触发作为拐点，后续只做确认
first_tp = df['turning_point_60'].idxmax()  # 第一个信号
# 然后在这个日期附近标记一个信号区间